# Hypervector Storage

`InMemory` provides ephemeral storage for notebooks — data disappears when the
Python process exits. `Fjall` provides persistent disk-backed storage. Both
implement the same `Memory` trait as Go's `badger.Memory`, with `mem_set` /
`mem_get` exposing the producer/selector interface. Convenience methods `put` /
`get` / `delete` / `names` wrap the common patterns.

Use `export(source, dest)` to persist an InMemory session to Fjall.

In [ ]:
try:
    import kongming_rs
except ImportError:
    %pip install -q kongming-rs-hv

In [ ]:
from kongming_rs import hv, memory
from kongming_rs.api.v1 import hv_pb2 as api

## Create a store

In [ ]:
storage = memory.InMemory(api.MODEL_64K_8BIT)
storage.name, storage.store_id, storage.item_count()

## Memory-trait API: `mem_set(producer)` / `mem_get(selector)`

These mirror Go's `Memory.Set(ChunkProducer)` / `Memory.Get(ChunkSelector)`.

Producers and selectors are first-class objects — construct them with factory
functions like `write_chunk_producer()`, `by_item_key()`, etc.

In [ ]:
words = hv.Domain.from_name("words")
hello = hv.Sparkle.from_word(api.MODEL_64K_8BIT, words, "hello")

# mem_set takes a Producer — here a terminal producer that creates a sparkle-only chunk
prod = memory.terminal("words", "world")
prod

In [ ]:
returned = storage.mem_set(prod)
type(returned).__name__

In [ ]:
# mem_get takes a Selector — by_item_key selects a single item
results = storage.mem_get(memory.by_item_key("words", "world"))
len(results), type(results[0]).__name__

In [ ]:
# new_learner producer creates a fresh Learner via the Memory trait
lr_via_trait = storage.mem_set(memory.new_learner("words", "my_learner"))
type(lr_via_trait).__name__

## Convenience API: `put` / `get` / `names`

`put(hv)` extracts domain and pod from the HV itself — every type (Sparkle, Set,
Learner, etc.) carries its own domain/pod. It also guards against protected
domains (prefix `~`) and maintains a name registry so `names(domain)` can return
human-readable strings.

In [ ]:
fruits = hv.Domain.from_name("fruits")
apple  = hv.Sparkle.from_word(api.MODEL_64K_8BIT, fruits, "apple")
banana = hv.Sparkle.from_word(api.MODEL_64K_8BIT, fruits, "banana")
cherry = hv.Sparkle.from_word(api.MODEL_64K_8BIT, fruits, "cherry")

storage.put(apple,  note="Granny Smith")
storage.put(banana, note="Cavendish")
storage.put(cherry)

storage.item_count(), storage.names("fruits")

## Retrieve and verify

`get` returns the exact HyperBinary type that was stored.

In [ ]:
got_apple = storage.get("fruits", "apple")
hv.equal(got_apple, apple), type(got_apple).__name__

In [ ]:
storage.note("fruits", "apple"), storage.note("fruits", "cherry")

## Store composites

Sets and Octopuses round-trip through the store. Probe with `overlap` and `release`.

In [ ]:
fruit_set = hv.Set(fruits, hv.Pod.from_word("fruit_set"), apple, banana, cherry)
storage.put(fruit_set)

got_set = storage.get("fruits", "fruit_set")
type(got_set).__name__

In [ ]:
# Probe the set — unmasked() strips the marker, revealing bundled members
unmasked = got_set.unmasked()
hv.overlap(unmasked, apple), hv.overlap(unmasked, banana), hv.overlap(unmasked, cherry)

In [ ]:
# Store an Octopus (key-value structure)
octo = hv.Octopus(fruits, hv.Pod.from_word("color_map"), ["fruit", "color"], apple, cherry)
storage.put(octo)

got_octo = storage.get("fruits", "color_map")
fruit_key = hv.Sparkle.from_word(api.MODEL_64K_8BIT, hv.d0(), "fruit")
color_key = hv.Sparkle.from_word(api.MODEL_64K_8BIT, hv.d0(), "color")
hv.overlap(hv.release(got_octo, fruit_key), apple), hv.overlap(hv.release(got_octo, color_key), cherry)

## Attractors and near-neighbor search (NNS)

Composites (Set, Sequence, Octopus) encode their content with structural
transformations — markers, position bindings, key bindings. An **attractor**
is a selector that *reverses* this encoding, exposing the raw content:

| Attractor | Input | What it does |
|-----------|-------|--------------|
| `set_members(sel)` | a Set | strips the SET_MARKER |
| `sequence_member(sel, pos)` | a Sequence | strips SEQUENCE_MARKER + step^pos |
| `tentacle(sel, key)` | an Octopus | strips the key binding |

An attractor alone just transforms the code — it doesn't search for anything.
To actually **find stored items** matching that content, wrap the attractor in
`nns()` (near-neighbor search). NNS uses the associative index to find items
whose codes overlap with the attractor's output:

```
mem_get( nns( attractor( selector ) ) )
         │       │          └── picks the composite from storage
         │       └── strips structural encoding (marker, position, key)
         └── searches the index for stored items matching the exposed content
```

In [ ]:
# Add a non-member to show NNS selectivity
grape = hv.Sparkle.from_word(api.MODEL_64K_8BIT, fruits, "grape")
storage.put(grape)

# "Which stored items are members of fruit_set?"
results = storage.mem_get(memory.nns(memory.set_members(memory.by_item_key(fruits, "fruit_set"))))
print(f"Set members found: {len(results)}")
for r in results:
    print(f"  apple={hv.overlap(r, apple):3d}  banana={hv.overlap(r, banana):3d}  "
          f"cherry={hv.overlap(r, cherry):3d}  grape={hv.overlap(r, grape):3d}")

In [ ]:
# "What value is stored under key 'fruit' in color_map?"
results = storage.mem_get(memory.nns(memory.tentacle(memory.by_item_key(fruits, "color_map"), "fruit")))
print(f"Tentacle 'fruit' found: {len(results)}")
for r in results:
    print(f"  apple={hv.overlap(r, apple):3d}  cherry={hv.overlap(r, cherry):3d}")

## Learner workflow

Learners are mutable hypervectors that accumulate observations over time.
There are two ways to train them:

1. **Direct bundle** — retrieve the Learner, call `bundle()` in Python, put it back
2. **ClusterUpdater producer** — update the Learner in storage via the Memory trait,
   without pulling it into Python

In [ ]:
# --- Method 1: Direct bundle in Python ---

learner = hv.Learner(api.MODEL_64K_8BIT, fruits, hv.Pod.from_word("fruit_learner"))
storage.put(learner)

# Retrieve, observe apple 3 times, banana and cherry once each
lr = storage.get("fruits", "fruit_learner")
lr.bundle(apple)
lr.bundle(banana)
lr.bundle(cherry)
lr.bundle(apple)
lr.bundle(apple)
storage.put(lr, note="5 observations: apple x3, banana x1, cherry x1")

# Apple should dominate (~3/5), banana and cherry minor (~1/5)
lr2 = storage.get("fruits", "fruit_learner")
print("Method 1 — direct bundle:")
print(f"  apple  weight={lr2.weight(apple):.3f}")
print(f"  banana weight={lr2.weight(banana):.3f}")
print(f"  cherry weight={lr2.weight(cherry):.3f}")

In [ ]:
# --- Method 2: ClusterUpdater producer (update in storage) ---
#
# cluster_updater(learner_selector, observed_selector, multiple)
# reads the learner and observed item from storage, bundles the
# observation into the learner, and writes it back — all server-side.

# Create a fresh learner via the Memory trait
storage.mem_set(memory.new_learner("fruits", "fruit_learner_2"))

# Observe apple 3 times using cluster_updater with multiple=3
storage.mem_set(memory.cluster_updater(
    memory.by_item_key(fruits, "fruit_learner_2"),
    memory.by_item_key(fruits, "apple"),
    multiple=3,
))

# Observe banana once
storage.mem_set(memory.cluster_updater(
    memory.by_item_key(fruits, "fruit_learner_2"),
    memory.by_item_key(fruits, "banana"),
))

# Observe cherry once
storage.mem_set(memory.cluster_updater(
    memory.by_item_key(fruits, "fruit_learner_2"),
    memory.by_item_key(fruits, "cherry"),
))

# Same distribution: apple x3, banana x1, cherry x1
lr3 = storage.get(fruits, "fruit_learner_2")
print("Method 2 — cluster_updater producer:")
print(f"  apple  weight={lr3.weight(apple):.3f}")
print(f"  banana weight={lr3.weight(banana):.3f}")
print(f"  cherry weight={lr3.weight(cherry):.3f}")

## Cross-domain isolation

Different domains are independent namespaces.

In [ ]:
colors = hv.Domain.from_name("colors")
red = hv.Sparkle.from_word(api.MODEL_64K_8BIT, colors, "red")
storage.put(red)

storage.names("fruits"), storage.names("colors")

## Export to persistent storage

`export(source, dest)` copies all entries (items + index) from one store
to another via the Substrate scan/write interface.

In [ ]:
import tempfile, os
tmpdir = tempfile.mkdtemp()
path = os.path.join(tmpdir, "my_store")

dest = memory.Fjall(api.MODEL_64K_8BIT, path)
count = memory.export(storage, dest)
print(f"Exported {count} entries to disk")

# Verify data survived the export
got = dest.get(fruits, "apple")
hv.equal(got, apple)

In [ ]:
# Reopen from disk — data persists
del dest
reopened = memory.Fjall(api.MODEL_64K_8BIT, path)
got2 = reopened.get(fruits, "banana")
print(f"Reopened from disk: equal={hv.equal(got2, banana)}")

# Cleanup
import shutil
shutil.rmtree(tmpdir)

## Substrate Views

`new_view()` and `new_mutable_view()` expose the substrate layer directly,
giving you read/write access within a transactional view. Use them as context
managers — the view is automatically discarded on exit, releasing the lock.

- **`SubstrateView`** (read-only): `chunk_exists`, `read_chunk`
- **`MutableSubstrateView`** (read-write): all of the above plus `write_chunk`
- **`memory.first_picked(view, selector)`** — select the first matching chunk (mirrors Go's `memory.FirstPicked`)

`MutableSubstrateView` follows transaction semantics:
- **Clean exit** → auto-commit
- **Exception** → rollback (pending writes discarded)

You can also call `commit()` explicitly mid-block, or `discard()` to release
the lock outside of a `with` statement.

In [ ]:
# Write chunks via a mutable view (auto-committed on clean exit)
mango = hv.Sparkle.from_word(api.MODEL_64K_8BIT, fruits, "mango")
kiwi  = hv.Sparkle.from_word(api.MODEL_64K_8BIT, fruits, "kiwi")

with storage.new_mutable_view() as view:
    view.write_chunk(mango, note="via mutable view")
    view.write_chunk(kiwi)
    # auto-committed here

# Read them back via a read-only view
with storage.new_view() as view:
    print("mango exists:", view.chunk_exists("fruits", "mango"))
    got_mango = view.read_chunk("fruits", "mango")
    print("round-trip equal:", hv.equal(got_mango, mango))

In [ ]:
# Batch write a composite within a single view transaction
tropical_set = hv.Set(fruits, hv.Pod.from_word("tropical"), mango, kiwi, banana)

with storage.new_mutable_view() as view:
    view.write_chunk(tropical_set)
    # read-your-writes: chunk is visible within the same view before commit
    print("visible before exit:", view.chunk_exists("fruits", "tropical"))
    # auto-committed here

# Verify from a fresh read-only view
with storage.new_view() as view:
    got = view.read_chunk("fruits", "tropical")
    print("type:", type(got).__name__)
    unmasked = got.unmasked()
    print(f"mango={hv.overlap(unmasked, mango)}  kiwi={hv.overlap(unmasked, kiwi)}  banana={hv.overlap(unmasked, banana)}")

In [ ]:
# view.discard() releases the lock explicitly (without a context manager)
view = storage.new_view()
print("before:", view)
got = view.read_chunk("fruits", "apple")
print("read apple:", hv.equal(got, apple))
view.discard()
print("after: ", view)

In [ ]:
# first_picked(view, selector): free function, mirrors Go's memory.FirstPicked(ctx, view, sel)
with storage.new_view() as view:
    got = memory.first_picked(view, memory.by_item_key("fruits", "cherry"))
    print("first_picked cherry:", type(got).__name__, hv.equal(got, cherry))

    # also works with NNS attractors
    found = memory.first_picked(view, memory.nns(memory.set_members(memory.by_item_key(fruits, "fruit_set"))))
    print("first NNS match from fruit_set:", type(found).__name__)